### Question 1
Load the Iris dataset using scikit-learn and use cross_val_score() with a KNeighborsClassifier to compute the average accuracy using 5-fold cross-validation.

Cross-validation partitions the dataset into $k=5$ disjoint folds, training the `KNeighborsClassifier` iteratively on 4 folds while validating on the remaining fold. Averaging the validation accuracies across all five iterations yields an estimated generalization accuracy of **97.33%** (individual fold scores: $0.9667$, $1.0000$, $0.9333$, $0.9667$, $1.0000$).

In [1]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# 1. Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# 2. Initialize KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)

# 3. Run 5-fold cross-validation
scores_5fold = cross_val_score(knn, X_iris, y_iris, cv=5, scoring='accuracy')

# 4. Display results
print("5-Fold Cross-Validation Scores for KNN:")
for i, score in enumerate(scores_5fold, 1):
    print(f"  Fold {i}: {score:.4f} ({score * 100:.2f}%)")

print(f"\nAverage 5-Fold Accuracy: {scores_5fold.mean():.4f} ({scores_5fold.mean() * 100:.2f}%)")
print(f"Score Standard Deviation: {scores_5fold.std():.4f}")

5-Fold Cross-Validation Scores for KNN:
  Fold 1: 0.9667 (96.67%)
  Fold 2: 1.0000 (100.00%)
  Fold 3: 0.9333 (93.33%)
  Fold 4: 0.9667 (96.67%)
  Fold 5: 1.0000 (100.00%)

Average 5-Fold Accuracy: 0.9733 (97.33%)
Score Standard Deviation: 0.0249


### Question 2
For a dataset of your choice (e.g., Titanic or any from scikit-learn), perform 10-fold cross-validation using cross_val_score() with a DecisionTreeClassifier and print the accuracy for each fold.

Using the Scikit-Learn Breast Cancer diagnostic dataset (569 samples, 30 features), a 10-fold cross-validation with `DecisionTreeClassifier(random_state=42)` evaluates how tree stability varies across sample subsets. The accuracy for each of the 10 folds ranges between **89.47%** and **98.25%**, producing a mean classification accuracy of **92.80%** with a standard deviation of $\pm 0.0337$.

In [2]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# 1. Load Breast Cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

# 2. Initialize DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)

# 3. Run 10-fold cross-validation
scores_10fold = cross_val_score(dt, X_cancer, y_cancer, cv=10, scoring='accuracy')

# 4. Display fold-by-fold results in a structured table
folds_df = pd.DataFrame({
    'Fold Index': [f"Fold {i}" for i in range(1, 11)],
    'Validation Accuracy': [f"{score:.4f} ({score * 100:.2f}%)" for score in scores_10fold]
})
display(folds_df)

print(f"Mean 10-Fold Accuracy:    {scores_10fold.mean():.4f} ({scores_10fold.mean() * 100:.2f}%)")
print(f"Standard Deviation:       {scores_10fold.std():.4f}")

,Fold Index,Validation Accuracy
0,Fold 1,0.9298 (92.98%)
1,Fold 2,0.8947 (89.47%)
2,Fold 3,0.9298 (92.98%)
3,Fold 4,0.8947 (89.47%)
4,Fold 5,0.9825 (98.25%)
5,Fold 6,0.8947 (89.47%)
6,Fold 7,0.8947 (89.47%)
7,Fold 8,0.9474 (94.74%)
8,Fold 9,0.9298 (92.98%)
9,Fold 10,0.9821 (98.21%)


Mean 10-Fold Accuracy:    0.9280 (92.80%)
Standard Deviation:       0.0327


### Question 3
Use StratifiedKFold from scikit-learn to split the Iris dataset and compare the distribution of target classes in each fold to show how stratified sampling works.

*(Hint: Print the class counts for each fold to verify stratification.)*

`StratifiedKFold` preserves the original target class proportions ($1:1:1$ ratio in Iris: 50 Setosa, 50 Versicolor, 50 Virginica) identically within every fold. With $k=5$ splits across 150 total samples, each validation fold contains exactly **10 samples of Class 0, 10 samples of Class 1, and 10 samples of Class 2**, preventing class imbalance or missing classes during cross-validation training and testing.

In [3]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# 1. Initialize StratifiedKFold with 5 splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Track class distribution in validation sets for each fold
stratification_data = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_iris, y_iris), 1):
    val_y = y_iris[val_idx]
    counts = np.bincount(val_y, minlength=len(iris.target_names))
    stratification_data.append({
        'Fold': f"Fold {fold}",
        f'Class 0 ({iris.target_names[0]})': counts[0],
        f'Class 1 ({iris.target_names[1]})': counts[1],
        f'Class 2 ({iris.target_names[2]})': counts[2],
        'Total Fold Samples': len(val_y),
        'Ratio (0:1:2)': f"{counts[0]}:{counts[1]}:{counts[2]}"
    })

# 3. Display summary table
stratification_df = pd.DataFrame(stratification_data)
print("StratifiedKFold Class Distribution Across All 5 Folds:")
display(stratification_df)
print("Verification: Every validation fold receives an identical 10:10:10 class representation, matching the true population distribution.")

StratifiedKFold Class Distribution Across All 5 Folds:


,Fold,Class 0 (setosa),Class 1 (versicolor),Class 2 (virginica),Total Fold Samples,Ratio (0:1:2)
0,Fold 1,10,10,10,30,10:10:10
1,Fold 2,10,10,10,30,10:10:10
2,Fold 3,10,10,10,30,10:10:10
3,Fold 4,10,10,10,30,10:10:10
4,Fold 5,10,10,10,30,10:10:10


Verification: Every validation fold receives an identical 10:10:10 class representation, matching the true population distribution.


### Question 4
Experiment with different values of 'cv' (3, 5, 10) in cross_val_score() for a RandomForestClassifier on the Wine dataset and report which value gives the most stable (least variable) accuracy scores.

*(Hint: Look at the standard deviation of the scores for each cv value.)*

Comparing cross-validation folds on the Wine dataset demonstrates that **`cv = 3` produces the most stable (least variable) accuracy scores**, exhibiting the lowest standard deviation ($\sigma = 0.0138$) compared to `cv = 5` ($\sigma = 0.0176$) and `cv = 10` ($\sigma = 0.0255$). While larger values of $k$ increase the size of the training partitions (lowering estimator bias), each validation split contains fewer samples ($17-18$ samples at $k=10$), making individual fold accuracies more volatile.

In [6]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 1. Load Wine dataset
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

# 2. Initialize Random Forest Classifier
rf = RandomForestClassifier(random_state=42)

# 3. Run cross-validation for cv = 3, 5, and 10
cv_options = [3, 5, 10]
stability_records = []

for cv_val in cv_options:
    scores = cross_val_score(rf, X_wine, y_wine, cv=cv_val, scoring='accuracy')
    stability_records.append({
        'cv Value': cv_val,
        'Mean Accuracy': f"{scores.mean() * 100:.2f}%",
        'Std Deviation': f"{scores.std():.4f}",
        'Score Range': f"{scores.min() * 100:.1f}% - {scores.max() * 100:.1f}%",
        'Fold Scores': list(scores.round(3))
    })

# Convert records directly to DataFrame
cv_comparison_df = pd.DataFrame(stability_records)

print("Random Forest Stability Across cv Folds (Wine Dataset):")
display(cv_comparison_df)
print("Conclusion: cv=3 delivers the lowest standard deviation, demonstrating the most stable accuracy spread across folds.")

Random Forest Stability Across cv Folds (Wine Dataset):


,cv Value,Mean Accuracy,Std Deviation,Score Range,Fold Scores
0,3,96.63%,0.0138,94.9% - 98.3%,"[0.967, 0.949, 0.983]"
1,5,97.21%,0.0176,94.4% - 100.0%,"[0.972, 0.944, 0.972, 0.971, 1.0]"
2,10,98.33%,0.0255,94.4% - 100.0%,"[0.944, 1.0, 0.944, 0.944, 1.0, 1.0, 1.0, 1.0,..."


Conclusion: cv=3 delivers the lowest standard deviation, demonstrating the most stable accuracy spread across folds.


### Question 5
Use ChatGPT or Copilot to help you write a function that takes any classifier and dataset, runs k-fold cross-validation using cross_val_score(), and returns the mean and standard deviation of the scores. Paste the code you generated and briefly describe how the AI tool improved your solution.

**AI-Generated Solution & Enhancements:**
The AI assistant generated a modular evaluation wrapper (`evaluate_model_cv`) and introduced three production enhancements: 
1. **Configurable Metric Scoring:** Added a `scoring` parameter with flexible defaults (e.g., `'accuracy'`, `'f1_macro'`, `'roc_auc'`).
2. **Parallel Processing:** Added `n_jobs=-1` to leverage multi-core CPU parallelism across folds.
3. **Structured Dictionary Output:** Formatted results as a clean dictionary containing the mean, standard deviation, and array of individual fold scores for automated logging.

In [9]:
from sklearn.model_selection import cross_val_score
import numpy as np

# AI-Generated and Enhanced Cross-Validation Evaluation Function
def evaluate_model_cv(classifier, X, y, cv=5, scoring='accuracy', n_jobs=-1):
    """
    Runs k-fold cross validation for any estimator and returns performance statistics.
    
    Parameters:
        classifier: Scikit-learn compatible estimator
        X: Feature matrix
        y: Target vector
        cv (int): Number of folds (default 5)
        scoring (str): Evaluation metric (default 'accuracy')
        n_jobs (int): CPU cores to use (-1 for all cores)
        
    Returns:
        dict: Mean score, standard deviation, and individual fold scores
    """
    scores = cross_val_score(classifier, X, y, cv=cv, scoring=scoring, n_jobs=n_jobs)
    mean_score = float(np.mean(scores))
    std_score = float(np.std(scores))
    
    return {
        'mean': mean_score,
        'std': std_score,
        'scores': scores.tolist()
    }

# Test the function across multiple models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Support Vector Machine': SVC(random_state=42)
}

print("Automated Benchmark via evaluate_model_cv (Iris Dataset, 5-Fold):\n")
benchmark_results = []
for name, model in models.items():
    res = evaluate_model_cv(model, X_iris, y_iris, cv=5)
    benchmark_results.append({
        'Model': name,
        'Mean Accuracy': f"{res['mean'] * 100:.2f}%",
        'Std Deviation': f"{res['std']:.4f}",
        'Scores': [round(s, 3) for s in res['scores']]
    })

display(pd.DataFrame(benchmark_results))

Automated Benchmark via evaluate_model_cv (Iris Dataset, 5-Fold):



,Model,Mean Accuracy,Std Deviation,Scores
0,K-Nearest Neighbors,97.33%,0.0249,"[0.967, 1.0, 0.933, 0.967, 1.0]"
1,Decision Tree,95.33%,0.0340,"[0.967, 0.967, 0.9, 0.933, 1.0]"
2,Random Forest,96.67%,0.0211,"[0.967, 0.967, 0.933, 0.967, 1.0]"
3,Support Vector Machine,96.67%,0.0211,"[0.967, 0.967, 0.967, 0.933, 1.0]"
